In [ ]:
# Install dependencies into the active kernel
%pip install flatten_json pandas matplotlib

In [ ]:
import pandas as pd
import json
from collections import defaultdict
from flatten_json import flatten
import matplotlib.pyplot as plt
import pathlib

In [ ]:
files = [
    'a_deep_analysis.json',
    'b_deep_analysis.json',
]

In [ ]:
def aggregate_loops_passes(loops):
    results_per_frame = []
    num_loops = len(loops)
    for loop_results in loops:
        for frame_index, frame_results in enumerate(loop_results["per_frame_results"]):
            if frame_index >= len(results_per_frame):
                results_per_frame.append(defaultdict(int))
            results_per_frame[frame_index]['sequence_time_ns'] = frame_results['sequence_time_ns']
            for command_buffer_timings in frame_results["command_buffer_timings"].values():
                for scope_name, scope_timings in command_buffer_timings["scope_timings"].items():
                    # A pass can run multiple times per frame; sum its durations, average over loops.
                    for scope_timing in scope_timings:
                        results_per_frame[frame_index][scope_name] += (
                            scope_timing["end"] - scope_timing["start"]
                        ) / num_loops / 1_000_000  # in ms
            if frame_results["metrics"] is not None:
                for metric_name, metric in frame_results["metrics"].items():
                    # TODO: Flatten this in rust to fan_speed_rpm
                    if metric is not None and metric_name != "timestamp":
                        results_per_frame[frame_index][metric_name] += metric / num_loops
    # TODO: Aggregate CPU timings
    return pd.DataFrame([flatten(x) for x in results_per_frame])


# Load every input file, aggregating its loops/passes into one number per pass per frame.
results = {}
for path in files:
    with open(path, "r") as json_file:
        results[path] = aggregate_loops_passes(json.load(json_file))

# Concat into one frame: (input file, frame) per row, metric per column
full_dataset = pd.concat(results)
full_dataset

In [ ]:
# Print all possible metrics
full_dataset.columns.tolist()

In [ ]:
metrics = full_dataset

# Reshape into sequence time + metric type per row, input file per column
metrics = metrics.reset_index().set_index(['sequence_time_ns', 'level_0']).drop('level_1', axis=1)
metrics = metrics.stack().unstack(1).reset_index()

# From ns to s
metrics['sequence_time_s'] = metrics['sequence_time_ns'] / 1_000_000_000
metrics = metrics.drop('sequence_time_ns', axis=1)
metrics

In [ ]:
pathlib.Path('output_analysis').mkdir(parents=True, exist_ok=True)

# Keep the N biggest passes individually and roll everything else into "other",
# so the stackplot stays readable instead of drowning in ~94 tiny bands.
TOP_N = 20

# Plot all metrics stacked on top of each other, one stackplot per input file,
# so we get a nice overview of how the total time is composed over the benchmark.
for file_name in files:
    file_data = full_dataset.loc[file_name]

    # x axis: benchmark timeline in seconds
    x = file_data['sequence_time_ns'] / 1_000_000_000

    # All metric columns (everything except the timeline), interpolated and gap-filled
    metric_cols = [c for c in file_data.columns if c != 'sequence_time_ns']
    stack_data = (
        file_data[metric_cols]
        .infer_objects(copy=False)
        .interpolate(method='linear')
        .fillna(0)
    )

    # Order by mean contribution, keep the top N, bucket the rest into "other"
    order = stack_data.mean().sort_values(ascending=False).index.tolist()
    top, rest = order[:TOP_N], order[TOP_N:]
    plot_data = stack_data[top].copy()
    if rest:
        plot_data[f'other ({len(rest)} passes)'] = stack_data[rest].sum(axis=1)

    # Biggest contributors at the bottom, "other" on top; distinct color per band
    labels = list(plot_data.columns)
    colors = plt.get_cmap('tab20')(range(len(labels)))

    fig, ax = plt.subplots(figsize=(20, 10))
    ax.stackplot(x, plot_data.T.values, labels=labels, colors=colors)
    ax.set_title(f'{file_name} - all metrics stacked (top {TOP_N} + other)')
    ax.set_xlabel('benchmark timeline in seconds')
    ax.set_ylabel('shader execution time in ms')
    ax.set_xlim(x.min(), x.max())
    ax.margins(y=0)
    ax.grid(True)

    # Reverse the legend so its order matches the top-to-bottom order of the stack
    handles, leg_labels = ax.get_legend_handles_labels()
    ax.legend(handles[::-1], leg_labels[::-1], loc='upper left', ncol=2, fontsize='small')

    fig.tight_layout()
    fig.savefig(f'output_analysis/{file_name}_stackplot.png')
